# Forest Fire AI — Final Tabular Model Training
## Notebook 08: Train Best Classifier & Regressor, Save All Artifacts


In [ ]:
import os, json, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, mean_absolute_error,
                              mean_squared_error, r2_score, confusion_matrix)
import xgboost as xgb

ROOT      = Path(r"e:/Sharvayu data/Malware/Symbiosis Nagpur SIT/7th SEM/Forest Fire task")
IMPL      = ROOT / "Implementation"
META_DIR  = IMPL / "artifacts" / "metadata"
PLOTS     = IMPL / "artifacts" / "plots"
DATA_DIR  = IMPL / "data" / "processed"
TAB_DIR   = IMPL / "models" / "tabular"
TAB_DIR.mkdir(parents=True, exist_ok=True)

with open(META_DIR / "tabular_metadata.json") as f:
    tmeta = json.load(f)
with open(META_DIR / "tabular_benchmark.json") as f:
    bench = json.load(f)

BEST_CLS = bench['best_classifier']
BEST_REG = bench['best_regressor']

df = pd.read_csv(DATA_DIR / "forestfires_processed.csv")
print(f"Data: {df.shape}  |  Best classifier: {BEST_CLS}  |  Best regressor: {BEST_REG}")


In [ ]:
# Feature engineering
num_feats = tmeta['numerical_features']

le_month = LabelEncoder()
le_day   = LabelEncoder()
df['month_enc'] = le_month.fit_transform(df['month'])
df['day_enc']   = le_day.fit_transform(df['day'])

FEATURE_COLS = num_feats + ['month_enc', 'day_enc']
X = df[FEATURE_COLS].values
y_cls = df['fire_occurred'].values
y_reg = np.log1p(df['area'].values)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

SEED = 42
X_tr, X_te, yc_tr, yc_te, yr_tr, yr_te = train_test_split(
    X, y_cls, y_reg, test_size=0.20, random_state=SEED, stratify=y_cls
)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

print(f"Train: {X_tr.shape}, Test: {X_te.shape}")
print(f"Fire in train: {yc_tr.sum()}, Fire in test: {yc_te.sum()}")


In [ ]:
# ── Train best classifier ─────────────────────────────────────────────
def get_classifier(name):
    if name == 'XGBoost':
        return xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                   subsample=0.8, colsample_bytree=0.8,
                                   random_state=SEED, eval_metric='logloss', verbosity=0)
    elif name == 'Random Forest':
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
    elif name == 'HistGradientBoosting':
        from sklearn.ensemble import HistGradientBoostingClassifier
        return HistGradientBoostingClassifier(max_iter=200, max_depth=5, random_state=SEED)
    else:
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(random_state=SEED, max_iter=2000)

clf = get_classifier(BEST_CLS)
t0 = time.time()
clf.fit(X_tr, yc_tr)
print(f"Classifier trained in {time.time()-t0:.1f}s")

yc_pred = clf.predict(X_te)
yc_prob = clf.predict_proba(X_te)[:, 1] if hasattr(clf, 'predict_proba') else yc_pred

acc   = accuracy_score(yc_te, yc_pred)
prec  = precision_score(yc_te, yc_pred, zero_division=0)
rec   = recall_score(yc_te, yc_pred, zero_division=0)
f1    = f1_score(yc_te, yc_pred, zero_division=0)
auc   = roc_auc_score(yc_te, yc_prob)

print(f"\nClassification Test Metrics:")
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1:        {f1:.4f}")
print(f"  ROC-AUC:   {auc:.4f}")


In [ ]:
# ── Train best regressor ──────────────────────────────────────────────
def get_regressor(name):
    if name == 'XGBoost':
        return xgb.XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=SEED, verbosity=0)
    elif name == 'Random Forest':
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
    else:
        from sklearn.ensemble import HistGradientBoostingRegressor
        return HistGradientBoostingRegressor(max_iter=200, max_depth=5, random_state=SEED)

reg = get_regressor(BEST_REG)
t0 = time.time()
reg.fit(X_tr, yr_tr)
print(f"Regressor trained in {time.time()-t0:.1f}s")

yr_pred_log = reg.predict(X_te)
yr_pred     = np.expm1(yr_pred_log)
yr_te_orig  = np.expm1(yr_te)

mae  = mean_absolute_error(yr_te_orig, yr_pred)
rmse = np.sqrt(mean_squared_error(yr_te_orig, yr_pred))
r2   = r2_score(yr_te, yr_pred_log)

print(f"\nRegression Test Metrics (on log-transformed target):")
print(f"  MAE (ha):  {mae:.4f}")
print(f"  RMSE (ha): {rmse:.4f}")
print(f"  R² (log):  {r2:.4f}")


In [ ]:
# ── Save all artifacts ────────────────────────────────────────────────
pickle.dump(clf,    open(TAB_DIR / "classifier.pkl",  "wb"))
pickle.dump(reg,    open(TAB_DIR / "regressor.pkl",   "wb"))
pickle.dump(scaler, open(TAB_DIR / "scaler.pkl",      "wb"))
pickle.dump(le_month, open(TAB_DIR / "le_month.pkl",  "wb"))
pickle.dump(le_day,   open(TAB_DIR / "le_day.pkl",    "wb"))

tabular_metadata = {
    'classifier_name': BEST_CLS,
    'regressor_name': BEST_REG,
    'feature_cols': FEATURE_COLS,
    'numerical_features': num_feats,
    'categorical_features': ['month', 'day'],
    'month_classes': list(le_month.classes_),
    'day_classes': list(le_day.classes_),
    'cls_metrics': {'accuracy': round(acc,4), 'precision': round(prec,4),
                    'recall': round(rec,4), 'f1': round(f1,4), 'roc_auc': round(auc,4)},
    'reg_metrics': {'mae': round(mae,4), 'rmse': round(rmse,4), 'r2': round(r2,4)}
}
with open(TAB_DIR / "metadata.json", "w") as f:
    json.dump(tabular_metadata, f, indent=2)

print("\nSaved artifacts:")
for art in ["classifier.pkl","regressor.pkl","scaler.pkl","le_month.pkl","le_day.pkl","metadata.json"]:
    print(f"  {TAB_DIR}/{art}")


In [ ]:
# Feature importance
if hasattr(clf, 'feature_importances_'):
    fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': clf.feature_importances_})
    fi = fi.sort_values('importance', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(fi['feature'], fi['importance'], color='#2E8B57', edgecolor='black', alpha=0.8)
    ax.set_title(f'Feature Importance — {BEST_CLS} Classifier', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(PLOTS / "tabular_feature_importance.png", dpi=100, bbox_inches='tight')
    plt.close()
    print("Feature importance plot saved.")
    fi_dict = fi.sort_values('importance', ascending=False).set_index('feature')['importance'].to_dict()
    with open(META_DIR / "feature_importance.json", "w") as f:
        json.dump({k: round(float(v), 6) for k, v in fi_dict.items()}, f, indent=2)

print("\nNotebook 08 complete.")
